In [ ]:
# 01. Data Inventory and Quality Audit

This notebook performs an initial audit of the datasets used in this project.

## Objectives

- inventory raw files across datasets
- summarize dataset size and yearly coverage
- evaluate schema consistency
- assess datetime coverage and missingness
- identify major data quality issues before modeling

## Datasets

- MDOT CHART event logs
- Maryland crash reports
- Lane- and zone-level detector readings

## Output

This notebook generates summary tables that guide downstream cleaning and modeling workflows.

In [ ]:
## 1. Configuration

In [2]:
from pathlib import Path

RAW_BASE = Path.cwd().parentPROJECT_ROOT = Path(r"\\")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print("RAW_BASE exists:", RAW_BASE.exists())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

RAW_BASE exists: True
PROJECT_ROOT exists: True


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import csv
import math
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

# 원본 데이터 위치 (OneDrive)
RAW_BASE = Path(r"")

# 결과 저장할 프로젝트 폴더
PROJECT_ROOT = Path(r"Traffic_Incident")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHART_DIR = RAW_BASE / "MDOT_CHART"
CRASH_DIR = RAW_BASE / "Crash_Report"
DETECTOR_DIRS = [
    RAW_BASE / "Detector_year_2022",
    RAW_BASE / "Detector_year_2023",
    RAW_BASE / "Detector_year_2024",
]

# detector는 샘플만 읽기
DETECTOR_SAMPLE_ROWS = 1000
GENERAL_SAMPLE_ROWS = 100000

DATETIME_CANDIDATES = [
    "Start time",
    "Closed time",
    "measurement_start",
    "Measurement Start",
    "start_time",
    "closed_time",
    "timestamp",
    "date_time",
    "datetime",
]

DETECTOR_CORE_COLS = [
    "zone_id",
    "lane_number",
    "lane_id",
    "measurement_start",
    "speed",
    "volume",
    "occupancy",
    "quality",
]

print("RAW_BASE exists:", RAW_BASE.exists())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("CHART_DIR exists:", CHART_DIR.exists())
print("CRASH_DIR exists:", CRASH_DIR.exists())
for d in DETECTOR_DIRS:
    print(f"{d.name} exists:", d.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

RAW_BASE exists: True
PROJECT_ROOT exists: True
CHART_DIR exists: True
CRASH_DIR exists: True
Detector_year_2022 exists: True
Detector_year_2023 exists: True
Detector_year_2024 exists: True
OUTPUT_DIR: \


In [ ]:
## 2. Helper Functions

In [6]:
# ============================================================
# HELPERS
# ============================================================

def read_csv_safe(path, nrows=None, usecols=None):
    try:
        return pd.read_csv(
            path,
            nrows=nrows,
            usecols=usecols,
            low_memory=False,
        )
    except UnicodeDecodeError:
        return pd.read_csv(
            path,
            nrows=nrows,
            usecols=usecols,
            encoding="latin1",
            low_memory=False,
        )


def list_csv_files(folder):
    if not folder.exists():
        return []
    return sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() == ".csv"
    ])


def infer_dataset_type(path):
    text = str(path).lower()
    name = path.name.lower()

    if "mdot_chart" in text or "event" in name:
        return "chart"
    if "crash_report" in text or "crash" in name:
        return "crash"
    if "detector" in text or "lane_readings" in name or "zone_readings" in name:
        return "detector"
    return "unknown"


def infer_year(path):
    m = re.search(r"(20\d{2})", str(path))
    return int(m.group(1)) if m else np.nan


def existing_col(columns, candidates):
    mapping = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in mapping:
            return mapping[key]
    return None


def detect_datetime_columns(columns):
    found = []
    mapping = {str(c).strip().lower(): c for c in columns}
    for cand in DATETIME_CANDIDATES:
        key = cand.strip().lower()
        if key in mapping:
            found.append(mapping[key])
    return found


def estimate_rows_from_sample(path, sample_lines=10000):
    """
    detector 파일처럼 많은 CSV에 대해 빠른 대략 row 수 추정
    """
    try:
        size = path.stat().st_size
        with open(path, "rb") as f:
            header = f.readline()
            bytes_read = 0
            lines = 0

            for _ in range(sample_lines):
                line = f.readline()
                if not line:
                    break
                bytes_read += len(line)
                lines += 1

        if lines == 0:
            return 0, "estimated_empty"

        avg_bytes_per_line = bytes_read / lines
        est_rows = int(max((size - len(header)) / avg_bytes_per_line, 0))
        return est_rows, "estimated_from_sample"

    except Exception:
        return np.nan, "estimate_failed"


def count_rows_exact(path):
    try:
        with open(path, "rb") as f:
            return max(sum(1 for _ in f) - 1, 0), "exact"
    except Exception:
        return np.nan, "count_failed"


def read_header_columns(path):
    try:
        df = read_csv_safe(path, nrows=0)
        return list(df.columns), None
    except Exception as e:
        return [], str(e)


def parse_csv_line(line):
    try:
        return next(csv.reader([line]))
    except Exception:
        return []


def get_last_nonempty_line(path, block_size=8192):
    with open(path, "rb") as f:
        f.seek(0, 2)
        file_size = f.tell()

        buffer = b""
        position = file_size

        while position > 0:
            read_size = min(block_size, position)
            position -= read_size
            f.seek(position)
            buffer = f.read(read_size) + buffer
            lines = buffer.splitlines()

            if len(lines) >= 2:
                for line in reversed(lines):
                    if line.strip():
                        return line.decode("utf-8-sig", errors="replace")

        return ""


def fast_first_last_datetime(path, time_col):
    """
    detector 파일이 시간순 정렬되어 있다고 가정하고
    첫 줄/마지막 줄만 이용해 시간 범위 추정
    """
    try:
        with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
            reader = csv.reader(f)
            header = next(reader)
            first_row = next(reader, None)

        if first_row is None:
            return pd.NaT, pd.NaT, "no_data_rows"

        if time_col not in header:
            return pd.NaT, pd.NaT, "time_col_not_found"

        idx = header.index(time_col)
        first_value = first_row[idx] if idx < len(first_row) else None

        last_line = get_last_nonempty_line(path)
        last_row = parse_csv_line(last_line)
        last_value = last_row[idx] if idx < len(last_row) else None

        first_dt = pd.to_datetime(first_value, errors="coerce")
        last_dt = pd.to_datetime(last_value, errors="coerce")

        return first_dt, last_dt, "fast_first_last"

    except Exception as e:
        return pd.NaT, pd.NaT, f"error: {e}"


def safe_missing_rate(df, col):
    if col not in df.columns:
        return np.nan
    return float(df[col].isna().mean())


def safe_zero_rate(df, col):
    if col not in df.columns:
        return np.nan
    x = pd.to_numeric(df[col], errors="coerce")
    return float((x == 0).mean())


def safe_nunique(df, col):
    if col not in df.columns:
        return np.nan
    return int(df[col].nunique(dropna=True))


def file_kind_detector(path):
    name = path.name.lower()
    if name.startswith("zone_readings"):
        return "zone_readings"
    if name.startswith("lane_readings"):
        return "lane_readings"
    return "detector_other"

In [ ]:
## 3. File Inventory

This section collects all raw CSV files and organizes them by dataset group.

In [7]:
chart_files = list_csv_files(CHART_DIR)
crash_files = list_csv_files(CRASH_DIR)

detector_files = []
for d in DETECTOR_DIRS:
    detector_files.extend(list_csv_files(d))

all_files = chart_files + crash_files + detector_files

print(f"CHART files: {len(chart_files):,}")
print(f"Crash files: {len(crash_files):,}")
print(f"Detector files: {len(detector_files):,}")
print(f"All files: {len(all_files):,}")

CHART files: 5
Crash files: 4
Detector files: 4,045
All files: 4,054


In [ ]:
## 4. Raw File Inventory

This section summarizes:
- file names
- file sizes
- approximate row counts
- available columns

In [ ]:
## 5. Dataset Summary

This section aggregates file inventory by dataset and year.

In [8]:
inventory_rows = []

for i, fp in enumerate(all_files, start=1):
    dataset_type = infer_dataset_type(fp)
    size_mb = fp.stat().st_size / 1024 / 1024
    columns, read_error = read_header_columns(fp)

    if dataset_type in ["chart", "crash"]:
        n_rows, row_count_method = count_rows_exact(fp)
    else:
        n_rows, row_count_method = estimate_rows_from_sample(fp)

    inventory_rows.append({
        "dataset_type": dataset_type,
        "year_inferred": infer_year(fp),
        "file_name": fp.name,
        "relative_path": str(fp.relative_to(RAW_BASE)),
        "size_mb": size_mb,
        "n_rows_approx": n_rows,
        "row_count_method": row_count_method,
        "n_columns": len(columns),
        "columns": ", ".join(map(str, columns)),
        "read_error": read_error,
    })

    if i % 500 == 0:
        print(f"Inventory processed: {i:,}/{len(all_files):,}")

inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(OUTPUT_DIR / "raw_file_inventory.csv", index=False)

dataset_summary = (
    inventory_df
    .groupby(["dataset_type", "year_inferred"], dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_size_mb=("size_mb", "sum"),
        total_rows_approx=("n_rows_approx", "sum"),
        median_file_size_mb=("size_mb", "median"),
    )
    .reset_index()
    .sort_values(["dataset_type", "year_inferred"])
)

dataset_summary.to_csv(OUTPUT_DIR / "mvp1_dataset_summary.csv", index=False)

print("Saved raw_file_inventory.csv")
print("Saved mvp1_dataset_summary.csv")
dataset_summary

Inventory processed: 500/4,054
Inventory processed: 1,000/4,054
Inventory processed: 1,500/4,054
Inventory processed: 2,000/4,054
Inventory processed: 2,500/4,054
Inventory processed: 3,000/4,054
Inventory processed: 3,500/4,054
Inventory processed: 4,000/4,054
Saved raw_file_inventory.csv
Saved mvp1_dataset_summary.csv


,dataset_type,year_inferred,n_files,total_size_mb,total_rows_approx,median_file_size_mb
0,chart,2020,1,16.592516,96619,16.592516
1,chart,2021,1,17.539365,103839,17.539365
2,chart,2022,1,17.349630,102357,17.349630
3,chart,2023,1,18.706389,108964,18.706389
4,chart,2024,1,18.294187,106217,18.294187
5,crash,2020,1,40.103875,476873,40.103875
6,crash,2021,1,46.199854,550662,46.199854
7,crash,2022,1,47.038424,552933,47.038424
8,crash,2023,1,48.360079,563775,48.360079
9,detector,2022,1373,11099.922655,172974493,5.868386


In [ ]:
## 6. Schema Audit

This section checks column consistency across files and years.

In [9]:
schema_rows = []

for _, row in inventory_df.iterrows():
    cols = str(row["columns"]).split(", ") if pd.notna(row["columns"]) else []
    for col in cols:
        if col:
            schema_rows.append({
                "dataset_type": row["dataset_type"],
                "year_inferred": row["year_inferred"],
                "file_name": row["file_name"],
                "column": col,
            })

schema_df = pd.DataFrame(schema_rows)

schema_summary = (
    schema_df
    .groupby(["dataset_type", "column"], dropna=False)
    .agg(
        n_files=("file_name", "nunique"),
        years=("year_inferred", lambda x: ", ".join(map(str, sorted(set(x.dropna()))))),
        example_files=("file_name", lambda x: "; ".join(sorted(set(x))[:5])),
    )
    .reset_index()
    .sort_values(["dataset_type", "n_files", "column"], ascending=[True, False, True])
)

schema_summary.to_csv(OUTPUT_DIR / "schema_summary.csv", index=False)

print("Saved schema_summary.csv")
schema_summary.head(30)

Saved schema_summary.csv


,dataset_type,column,n_files,years,example_files
0,chart,Agency,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
1,chart,Agency-specific Sub Type,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
2,chart,Agency-specific Type,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
3,chart,Closed time,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
4,chart,Duration (Incident clearance time),5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
5,chart,Location,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
6,chart,Max Lanes Closed,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
7,chart,Op. Center,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
8,chart,Operator Notes,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...
9,chart,Responders,5,"2020, 2021, 2022, 2023, 2024",MDOT_CHART_2020.csv; MDOT_CHART_2021.csv; MDOT...


In [ ]:
## 7. Datetime Coverage

This section evaluates timestamp coverage across datasets.

In [10]:
datetime_rows = []

for i, fp in enumerate(all_files, start=1):
    dataset_type = infer_dataset_type(fp)
    columns, read_error = read_header_columns(fp)

    if read_error:
        continue

    dt_cols = detect_datetime_columns(columns)

    if dataset_type == "detector":
        time_col = existing_col(columns, ["measurement_start", "Measurement Start"])
        if time_col:
            min_dt, max_dt, method = fast_first_last_datetime(fp, time_col)
            datetime_rows.append({
                "dataset_type": dataset_type,
                "year_inferred": infer_year(fp),
                "file_name": fp.name,
                "datetime_column": time_col,
                "min_datetime": min_dt,
                "max_datetime": max_dt,
                "datetime_method": method,
                "datetime_null_rate": np.nan,
            })
    else:
        for dt_col in dt_cols:
            try:
                temp = read_csv_safe(fp, usecols=[dt_col])
                dt = pd.to_datetime(temp[dt_col], errors="coerce")

                datetime_rows.append({
                    "dataset_type": dataset_type,
                    "year_inferred": infer_year(fp),
                    "file_name": fp.name,
                    "datetime_column": dt_col,
                    "min_datetime": dt.min(),
                    "max_datetime": dt.max(),
                    "datetime_method": "full_column_scan",
                    "datetime_null_rate": float(dt.isna().mean()),
                })
            except Exception as e:
                datetime_rows.append({
                    "dataset_type": dataset_type,
                    "year_inferred": infer_year(fp),
                    "file_name": fp.name,
                    "datetime_column": dt_col,
                    "min_datetime": pd.NaT,
                    "max_datetime": pd.NaT,
                    "datetime_method": f"error: {e}",
                    "datetime_null_rate": np.nan,
                })

    if i % 500 == 0:
        print(f"Datetime processed: {i:,}/{len(all_files):,}")

datetime_df = pd.DataFrame(datetime_rows)
datetime_df.to_csv(OUTPUT_DIR / "datetime_range_summary.csv", index=False)

print("Saved datetime_range_summary.csv")
datetime_df.head(20)

Datetime processed: 500/4,054
Datetime processed: 1,000/4,054
Datetime processed: 1,500/4,054
Datetime processed: 2,000/4,054
Datetime processed: 2,500/4,054
Datetime processed: 3,000/4,054
Datetime processed: 3,500/4,054
Datetime processed: 4,000/4,054
Saved datetime_range_summary.csv


,dataset_type,year_inferred,file_name,datetime_column,min_datetime,max_datetime,datetime_method,datetime_null_rate
0,chart,2020,MDOT_CHART_2020.csv,Start time,2020-01-01 00:00:00-05:00,2020-12-31 23:59:00-05:00,full_column_scan,0.0
1,chart,2020,MDOT_CHART_2020.csv,Closed time,2020-01-01 00:47:00-05:00,2025-03-04 03:50:00-05:00,full_column_scan,0.0
2,chart,2021,MDOT_CHART_2021.csv,Start time,2021-01-01 00:00:00-05:00,2021-12-31 23:47:00-05:00,full_column_scan,0.0
3,chart,2021,MDOT_CHART_2021.csv,Closed time,2021-01-01 00:33:00-05:00,2025-05-23 11:10:00-04:00,full_column_scan,0.0
4,chart,2022,MDOT_CHART_2022.csv,Start time,2022-01-01 00:00:00-05:00,2022-12-31 23:58:00-05:00,full_column_scan,0.0
5,chart,2022,MDOT_CHART_2022.csv,Closed time,NaT,NaT,error: '<=' not supported between instances of...,NaN
6,chart,2023,MDOT_CHART_2023.csv,Start time,2023-01-01 00:00:00-05:00,2023-12-31 23:53:00-05:00,full_column_scan,0.0
7,chart,2023,MDOT_CHART_2023.csv,Closed time,NaT,NaT,error: '<=' not supported between instances of...,NaN
8,chart,2024,MDOT_CHART_2024.csv,Start time,2024-01-01 00:00:00-05:00,2024-12-31 23:49:00-05:00,full_column_scan,0.0
9,chart,2024,MDOT_CHART_2024.csv,Closed time,NaT,NaT,error: '<=' not supported between instances of...,NaN


In [ ]:
## 8. Missingness Audit

This section evaluates sample-based missingness for key fields across files.

In [11]:
missing_rows = []

for i, fp in enumerate(all_files, start=1):
    dataset_type = infer_dataset_type(fp)

    try:
        nrows = DETECTOR_SAMPLE_ROWS if dataset_type == "detector" else GENERAL_SAMPLE_ROWS
        df = read_csv_safe(fp, nrows=nrows)

        for col in df.columns:
            missing_rows.append({
                "dataset_type": dataset_type,
                "year_inferred": infer_year(fp),
                "file_name": fp.name,
                "column": col,
                "sample_n": len(df),
                "missing_rate_sample": float(df[col].isna().mean()),
                "n_unique_sample": int(df[col].nunique(dropna=True)),
                "dtype_sample": str(df[col].dtype),
            })

    except Exception as e:
        missing_rows.append({
            "dataset_type": dataset_type,
            "year_inferred": infer_year(fp),
            "file_name": fp.name,
            "column": "READ_ERROR",
            "sample_n": np.nan,
            "missing_rate_sample": np.nan,
            "n_unique_sample": np.nan,
            "dtype_sample": str(e),
        })

    if i % 500 == 0:
        print(f"Missingness processed: {i:,}/{len(all_files):,}")

missing_df = pd.DataFrame(missing_rows)
missing_df.to_csv(OUTPUT_DIR / "missingness_sample_summary.csv", index=False)

print("Saved missingness_sample_summary.csv")
missing_df.head(20)

Missingness processed: 500/4,054
Missingness processed: 1,000/4,054
Missingness processed: 1,500/4,054
Missingness processed: 2,000/4,054
Missingness processed: 2,500/4,054
Missingness processed: 3,000/4,054
Missingness processed: 3,500/4,054
Missingness processed: 4,000/4,054
Saved missingness_sample_summary.csv


,dataset_type,year_inferred,file_name,column,sample_n,missing_rate_sample,n_unique_sample,dtype_sample
0,chart,2020,MDOT_CHART_2020.csv,Agency,96619,0.000000,1,object
1,chart,2020,MDOT_CHART_2020.csv,Standardized Type,96619,0.000000,20,object
2,chart,2020,MDOT_CHART_2020.csv,Agency-specific Type,96619,0.000000,9,object
3,chart,2020,MDOT_CHART_2020.csv,Agency-specific Sub Type,96619,1.000000,0,float64
4,chart,2020,MDOT_CHART_2020.csv,Start time,96619,0.000000,84110,object
5,chart,2020,MDOT_CHART_2020.csv,Closed time,96619,0.000000,77458,object
6,chart,2020,MDOT_CHART_2020.csv,Location,96619,0.000000,36590,object
7,chart,2020,MDOT_CHART_2020.csv,Op. Center,96619,0.000000,8,object
8,chart,2020,MDOT_CHART_2020.csv,Duration (Incident clearance time),96619,0.000000,9887,object
9,chart,2020,MDOT_CHART_2020.csv,Operator Notes,96619,0.000000,55,int64


In [ ]:
## 9. Detector Coverage Summary

This section summarizes detector file coverage and sample-level quality indicators.

In [12]:
detector_rows = []

for i, fp in enumerate(detector_files, start=1):
    try:
        columns, read_error = read_header_columns(fp)
        kind = file_kind_detector(fp)

        if read_error:
            detector_rows.append({
                "year_inferred": infer_year(fp),
                "file_name": fp.name,
                "relative_path": str(fp.relative_to(RAW_BASE)),
                "detector_file_kind": kind,
                "read_status": "header_error",
                "error": read_error,
            })
            continue

        usecols = [c for c in DETECTOR_CORE_COLS if c in columns]
        if not usecols:
            usecols = columns

        df = read_csv_safe(fp, nrows=DETECTOR_SAMPLE_ROWS, usecols=usecols)

        time_col = existing_col(columns, ["measurement_start", "Measurement Start"])
        min_dt, max_dt, dt_method = (
            fast_first_last_datetime(fp, time_col)
            if time_col else (pd.NaT, pd.NaT, "no_time_col")
        )

        n_rows_est, row_method = estimate_rows_from_sample(fp)

        if (
            kind == "lane_readings"
            and pd.notna(min_dt)
            and pd.notna(max_dt)
            and max_dt >= min_dt
        ):
            expected_5min = int(((max_dt - min_dt).total_seconds() / 300) + 1)
            coverage_ratio_approx = (
                n_rows_est / expected_5min
                if expected_5min > 0 else np.nan
            )
        else:
            expected_5min = np.nan
            coverage_ratio_approx = np.nan

        detector_rows.append({
            "year_inferred": infer_year(fp),
            "file_name": fp.name,
            "relative_path": str(fp.relative_to(RAW_BASE)),
            "detector_file_kind": kind,
            "size_mb": fp.stat().st_size / 1024 / 1024,
            "n_rows_approx": n_rows_est,
            "row_count_method": row_method,
            "n_columns": len(columns),
            "sample_n": len(df),

            "min_time_fast": min_dt,
            "max_time_fast": max_dt,
            "datetime_method": dt_method,
            "expected_5min_records_between_min_max": expected_5min,
            "coverage_ratio_approx": coverage_ratio_approx,

            "n_unique_zone_sample": safe_nunique(df, "zone_id"),
            "n_unique_lane_id_sample": safe_nunique(df, "lane_id"),
            "n_unique_lane_number_sample": safe_nunique(df, "lane_number"),

            "speed_missing_rate_sample": safe_missing_rate(df, "speed"),
            "volume_missing_rate_sample": safe_missing_rate(df, "volume"),
            "occupancy_missing_rate_sample": safe_missing_rate(df, "occupancy"),

            "speed_zero_rate_sample": safe_zero_rate(df, "speed"),
            "volume_zero_rate_sample": safe_zero_rate(df, "volume"),
            "occupancy_zero_rate_sample": safe_zero_rate(df, "occupancy"),

            "read_status": "ok",
        })

    except Exception as e:
        detector_rows.append({
            "year_inferred": infer_year(fp),
            "file_name": fp.name,
            "relative_path": str(fp.relative_to(RAW_BASE)),
            "detector_file_kind": file_kind_detector(fp),
            "read_status": "error",
            "error": str(e),
        })

    if i % 250 == 0:
        print(f"Detector inventory processed: {i:,}/{len(detector_files):,}")

detector_inventory = pd.DataFrame(detector_rows)
detector_inventory.to_csv(OUTPUT_DIR / "detector_file_inventory_fast.csv", index=False)

detector_ok = detector_inventory[detector_inventory["read_status"] == "ok"].copy()

detector_year_summary = (
    detector_ok
    .groupby(["year_inferred", "detector_file_kind"], dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_size_mb=("size_mb", "sum"),
        total_rows_approx=("n_rows_approx", "sum"),
        median_coverage_ratio_approx=("coverage_ratio_approx", "median"),
        median_speed_missing_sample=("speed_missing_rate_sample", "median"),
        median_volume_missing_sample=("volume_missing_rate_sample", "median"),
        median_occupancy_missing_sample=("occupancy_missing_rate_sample", "median"),
        median_speed_zero_sample=("speed_zero_rate_sample", "median"),
        median_occupancy_zero_sample=("occupancy_zero_rate_sample", "median"),
    )
    .reset_index()
    .sort_values(["year_inferred", "detector_file_kind"])
)

detector_year_summary.to_csv(OUTPUT_DIR / "detector_year_summary.csv", index=False)

print("Saved detector_file_inventory_fast.csv")
print("Saved detector_year_summary.csv")
detector_year_summary

Detector inventory processed: 250/4,045
Detector inventory processed: 500/4,045
Detector inventory processed: 750/4,045
Detector inventory processed: 1,000/4,045
Detector inventory processed: 1,250/4,045
Detector inventory processed: 1,500/4,045
Detector inventory processed: 1,750/4,045
Detector inventory processed: 2,000/4,045
Detector inventory processed: 2,250/4,045
Detector inventory processed: 2,500/4,045
Detector inventory processed: 2,750/4,045
Detector inventory processed: 3,000/4,045
Detector inventory processed: 3,250/4,045
Detector inventory processed: 3,500/4,045
Detector inventory processed: 3,750/4,045
Detector inventory processed: 4,000/4,045
Saved detector_file_inventory_fast.csv
Saved detector_year_summary.csv


,year_inferred,detector_file_kind,n_files,total_size_mb,total_rows_approx,median_coverage_ratio_approx,median_speed_missing_sample,median_volume_missing_sample,median_occupancy_missing_sample,median_speed_zero_sample,median_occupancy_zero_sample
0,2022,detector_other,1,0.236883,1672,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,lane_readings,1371,8252.924111,125826648,0.931649,0.001,0.0,0.0,0.0,0.014
2,2022,zone_readings,1,2846.761661,47146173,NaN,0.024,0.0,0.0,0.0,0.079
3,2023,detector_other,1,0.236883,1672,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,lane_readings,1489,9325.226965,133755881,0.659256,0.002,0.0,0.0,0.0,0.014
5,2023,zone_readings,1,3605.955273,59525761,NaN,0.019,0.0,0.0,0.0,0.078
6,2024,detector_other,1,0.236883,1672,NaN,NaN,NaN,NaN,NaN,NaN
7,2024,lane_readings,1180,15467.645951,218139652,0.923950,0.003,0.0,0.0,0.0,0.022


In [ ]:
## 10. CHART Duration Quality Summary

This section provides a quick overview of duration completeness and event coverage in the CHART data.

In [13]:
chart_quality_rows = []
chart_type_rows_all = []

for fp in chart_files:
    try:
        df = read_csv_safe(fp)

        start_col = existing_col(df.columns, ["Start time", "start_time"])
        closed_col = existing_col(df.columns, ["Closed time", "closed_time"])
        type_col = existing_col(df.columns, ["Standardized Type", "standardized_type"])

        if start_col is None or closed_col is None:
            chart_quality_rows.append({
                "file_name": fp.name,
                "year_inferred": infer_year(fp),
                "n_rows": len(df),
                "read_status": "missing_start_or_closed",
            })
            continue

        start = pd.to_datetime(df[start_col], errors="coerce")
        closed = pd.to_datetime(df[closed_col], errors="coerce")
        duration_min = (closed - start).dt.total_seconds() / 60

        chart_quality_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "n_rows": len(df),
            "start_missing_rate": float(start.isna().mean()),
            "closed_missing_rate": float(closed.isna().mean()),
            "duration_missing_rate": float(duration_min.isna().mean()),
            "duration_nonpositive_rate": float((duration_min <= 0).mean()),
            "duration_gt_24hr_rate": float((duration_min > 24 * 60).mean()),
            "duration_gt_7day_rate": float((duration_min > 7 * 24 * 60).mean()),
            "duration_median_min": float(duration_min.median()),
            "duration_p90_min": float(duration_min.quantile(0.90)),
            "duration_p99_min": float(duration_min.quantile(0.99)),
            "n_event_types": int(df[type_col].nunique(dropna=True)) if type_col else np.nan,
            "read_status": "ok",
        })

        if type_col is not None:
            temp = pd.DataFrame({
                "year_inferred": infer_year(fp),
                "event_type": df[type_col],
                "duration_min": duration_min,
            })
            chart_type_rows_all.append(temp)

    except Exception as e:
        chart_quality_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "read_status": "error",
            "error": str(e),
        })

chart_quality = pd.DataFrame(chart_quality_rows)
chart_quality.to_csv(OUTPUT_DIR / "chart_duration_quality_summary.csv", index=False)

if chart_type_rows_all:
    chart_type_df = pd.concat(chart_type_rows_all, ignore_index=True)
    chart_type_df = chart_type_df.dropna(subset=["event_type"])

    chart_event_type_summary = (
        chart_type_df
        .groupby(["event_type"], dropna=False)
        .agg(
            n_events=("duration_min", "size"),
            duration_median_min=("duration_min", "median"),
            duration_p75_min=("duration_min", lambda x: x.quantile(0.75)),
            duration_p90_min=("duration_min", lambda x: x.quantile(0.90)),
            duration_p99_min=("duration_min", lambda x: x.quantile(0.99)),
            nonpositive_rate=("duration_min", lambda x: float((x <= 0).mean())),
            gt_24hr_rate=("duration_min", lambda x: float((x > 24 * 60).mean())),
        )
        .reset_index()
        .sort_values("n_events", ascending=False)
    )
else:
    chart_event_type_summary = pd.DataFrame()

chart_event_type_summary.to_csv(
    OUTPUT_DIR / "chart_event_type_duration_summary.csv",
    index=False,
)

print("Saved chart_duration_quality_summary.csv")
print("Saved chart_event_type_duration_summary.csv")
chart_quality

Saved chart_duration_quality_summary.csv
Saved chart_event_type_duration_summary.csv


,file_name,year_inferred,read_status,error
0,MDOT_CHART_2020.csv,2020,error,Can only use .dt accessor with datetimelike va...
1,MDOT_CHART_2021.csv,2021,error,Can only use .dt accessor with datetimelike va...
2,MDOT_CHART_2022.csv,2022,error,Can only use .dt accessor with datetimelike va...
3,MDOT_CHART_2023.csv,2023,error,Can only use .dt accessor with datetimelike va...
4,MDOT_CHART_2024.csv,2024,error,Can only use .dt accessor with datetimelike va...


In [ ]:
## 11. Crash Record Structure Summary

This section summarizes record types and crash ID coverage in the crash datasets.

In [14]:
crash_rows = []

for fp in crash_files:
    try:
        df = read_csv_safe(fp)

        record_type_col = existing_col(df.columns, ["Record Type", "record_type", "recordType"])
        crash_id_col = existing_col(df.columns, ["Crash ID", "crash_id", "CrashId"])
        start_col = existing_col(df.columns, ["Start time", "start_time"])

        if start_col:
            start = pd.to_datetime(df[start_col], errors="coerce")
            min_start = start.min()
            max_start = start.max()
            start_missing_rate = float(start.isna().mean())
        else:
            min_start = pd.NaT
            max_start = pd.NaT
            start_missing_rate = np.nan

        if record_type_col:
            counts = df[record_type_col].value_counts(dropna=False)

            for record_type, n in counts.items():
                sub = df[df[record_type_col] == record_type]

                crash_rows.append({
                    "file_name": fp.name,
                    "year_inferred": infer_year(fp),
                    "record_type": record_type,
                    "n_rows": int(n),
                    "share": float(n / len(df)),
                    "n_unique_crash_ids": int(sub[crash_id_col].nunique(dropna=True)) if crash_id_col else np.nan,
                    "min_start_time": min_start,
                    "max_start_time": max_start,
                    "start_missing_rate_all_rows": start_missing_rate,
                    "read_status": "ok",
                })
        else:
            crash_rows.append({
                "file_name": fp.name,
                "year_inferred": infer_year(fp),
                "record_type": "NO_RECORD_TYPE_COLUMN",
                "n_rows": len(df),
                "share": 1.0,
                "n_unique_crash_ids": int(df[crash_id_col].nunique(dropna=True)) if crash_id_col else np.nan,
                "min_start_time": min_start,
                "max_start_time": max_start,
                "start_missing_rate_all_rows": start_missing_rate,
                "read_status": "ok",
            })

    except Exception as e:
        crash_rows.append({
            "file_name": fp.name,
            "year_inferred": infer_year(fp),
            "record_type": "READ_ERROR",
            "read_status": "error",
            "error": str(e),
        })

crash_summary = pd.DataFrame(crash_rows)
crash_summary.to_csv(OUTPUT_DIR / "crash_record_type_summary.csv", index=False)

print("Saved crash_record_type_summary.csv")
crash_summary.head(20)

Saved crash_record_type_summary.csv


,file_name,year_inferred,record_type,read_status,error
0,Crashes_2020.csv,2020,READ_ERROR,error,'<=' not supported between instances of 'datet...
1,Crashes_2021.csv,2021,READ_ERROR,error,'<=' not supported between instances of 'datet...
2,Crashes_2022.csv,2022,READ_ERROR,error,'<=' not supported between instances of 'datet...
3,Crashes_2023.csv,2023,READ_ERROR,error,'<=' not supported between instances of 'datet...


In [ ]:
## 12. Summary

The data audit confirms that the project datasets provide sufficient scale and structure for downstream modeling.

Key observations:

- CHART event logs provide strong multi-year operational coverage
- detector data require selective sampling and quality checks due to file volume
- crash data include multiple record types and require structured aggregation

These inventory outputs support the cleaning and modeling steps implemented in the next notebooks.